**IMPORT**

In [13]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import optuna
import warnings
import os
from huggingface_hub import HfApi, hf_hub_download
warnings.filterwarnings("ignore")

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("CatBoost_Optuna")
REPO_ID = "thtdung040209020945/CatBoost"
DB_FILE = "catboost_study.db"

api = HfApi(token = HF_TOKEN)
api.create_repo(repo_id="CatBoost", repo_type = "dataset", private = True)


HfHubHTTPError: Client error '409 Conflict' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-69c28ef1-05687f854c15f8807c955bfa;e18f146e-8cde-44b5-b11e-43515fc8c79a)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/409

You already created this dataset repo: thtdung040209020945/CatBoost

**DATA PREPROCESSING**# 

In [14]:
train = pd.read_csv('/kaggle/input/competitions/2026-ptnk-ai-predicting-heart-disease/train.csv')
test = pd.read_csv('/kaggle/input/competitions/2026-ptnk-ai-predicting-heart-disease/test.csv')
train['Heart Disease'] = train['Heart Disease'].map({'Presence': 1, 'Absence': 0})

TARGET = 'Heart Disease'
features = [col for col in train.columns if col not in ['id', TARGET]]


cat_features = ['Sex', 'Chest pain type', 'FBS over 120', 'EKG results', 
                'Exercise angina', 'Slope of ST', 'Thallium']

for col in cat_features:
    train[col] = train[col].astype(str)
    test[col] = test[col].astype(str)

**FEATURE ENGINEERING**

In [15]:
def feature_engineering(df):
    df_out = df.copy()

    # Rate Pressure Product --> khối lượng công việc mà tim đang phải gánh
    df_out['RPP'] = df_out['BP'] * df_out['Max HR']
    # Chỉ số nhịp tim tối đa thâm hụt --> Nếu Max HR thực tế < mức này quá nhiều --> Tim yếu 
    df_out['Max_HR_Deficit'] = (220 - df_out['Age']) - df_out['Max HR']

    # Huyết áp và cholesterol tăng theo tuổi --> nếu người nhỏ mà huyết áo / cholesterol cao giống người lớn tuổi thì nguy hiểm hơn
    df_out['BP_Age_Ratio'] = df_out['BP'] / (df_out['Age'] + 1e-5)
    df_out['Chol_Age_Ratio'] = df_out['Cholesterol'] / (df_out['Age'] + 1e-5)
    
    # Tổng Rủi ro tổng hợp (Severity Score)
    df_out['Severity_Score'] = (
        (df_out['FBS over 120'] == 1).astype(int) +
        (df_out['Exercise angina'] == 1).astype(int) +
        (df_out['ST depression'] > 1.0).astype(int) +
        (df_out['Number of vessels fluro'] > 0).astype(int)
    )
    
    # GBDT khó nhận diện tích/thương, ta tạo sẵn cho nó
    df_out['ST_mul_Slope'] = df_out['ST depression'] * df_out['Slope of ST'].astype(float)
    
    return df_out

train = feature_engineering(train)
test = feature_engineering(test)

**OPTUNA**

In [22]:
def load_existing_study_db():
    if os.path.exists(DB_FILE):
        os.remove(DB_FILE)
    try: 
        print(f"🔄 Đang kiểm tra Database cũ từ Hugging Face...")
        hf_hub_download(
            repo_id=REPO_ID,
            filename=DB_FILE,
            repo_type="dataset",
            local_dir=".",
            token=HF_TOKEN
        )
        print("✅ Đã tìm thấy và tải Database cũ. Sẽ RESUME quá trình tuning.")
    except Exception as e:
        print("⚠️ Không tìm thấy file DB trên Cloud. Sẽ tạo Database mới từ đầu.")

In [24]:
def save_study_callback(study, trial): # Callback tự động upload file .db lên HF sau mỗi Trial
    try:
        print(f"☁️ Đang đồng bộ trạng thái sau Trial {trial.number} lên HF...")
        api.upload_file(
            path_or_fileobj=DB_FILE,
            path_in_repo=DB_FILE,
            repo_id=REPO_ID,
            repo_type="dataset"
        )
        print("✅ Đã lưu tiến độ an toàn!")
    except Exception as e:
        print(f"❌ Lỗi khi upload backup: {e}")

In [ ]:
def objective(trial):
    param = {
        "iterations": 1000, 
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        
        # CHỈNH SỬA: Giới hạn độ sâu cây để tăng tính tổng quát hóa
        "depth": trial.suggest_int("depth", 3, 6), 
        
        # CHỈNH SỬA: Tăng cường L2 Regularization
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 3.0, 20.0, log=True), 
        
        # CHỈNH SỬA: Thêm độ ngẫu nhiên mạnh hơn
        "random_strength": trial.suggest_float("random_strength", 0.1, 10.0, log=True), 
        
        "border_count": trial.suggest_categorical("border_count", [32, 64, 128, 254]),
        "eval_metric": "AUC",
        "random_seed": 42,
        "verbose": False 
    }
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(train[features], train[TARGET])):
        X_tr, y_tr = train.iloc[train_idx][features], train.iloc[train_idx][TARGET]
        X_va, y_va = train.iloc[valid_idx][features], train.iloc[valid_idx][TARGET]

        model = CatBoostClassifier(**param, cat_features=cat_features)
        
        model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            early_stopping_rounds=50, # Ngừng sớm nếu AUC không tăng sau 50 vòng
            use_best_model=True
        )

        preds = model.predict_proba(X_va)[:, 1]
        auc = roc_auc_score(y_va, preds)
        cv_scores.append(auc)

    return np.mean(cv_scores)


load_existing_study_db()
storage_name = f"sqlite:///{DB_FILE}"

study = optuna.create_study(
    direction="maximize", # Mục tiêu là tối đa hóa ROC AUC
    study_name="catboost_heart_disease_opt", 
    storage=storage_name,
    load_if_exists=True # Cực kỳ quan trọng để resume
)

TOTAL_TRIALS = 50 
current_trials = len(study.trials)
print(f"📊 Trạng thái hiện tại: Đã chạy xong {current_trials} trials.")

remaining_trials = TOTAL_TRIALS - current_trials

if remaining_trials > 0:
    print(f"🚀 Bắt đầu chạy tiếp {remaining_trials} trials còn lại...")
    # Gắn hàm callback upload vào đây
    study.optimize(objective, n_trials=remaining_trials, callbacks=[save_study_callback])
else:
    print("🎉 Đã hoàn thành đủ số lượng Trials mục tiêu!")

# --- 4. IN KẾT QUẢ ---
print("\n" + "="*40)
print("🏆 KẾT QUẢ TỐI ƯU NHẤT")
print("="*40)
try:
    print(f"Best Trial AUC: {study.best_value:.5f}")
    print("Best Parameters:")
    for key, value in study.best_params.items():
        print(f"    {key}: {value}")
except ValueError:
    print("Chưa có Trial nào hoàn thành thành công.")

🔄 Đang kiểm tra Database cũ từ Hugging Face...


catboost_study.db:   0%|          | 0.00/131k [00:00<?, ?B/s]

✅ Đã tìm thấy và tải Database cũ. Sẽ RESUME quá trình tuning.


[I 2026-03-24 13:18:15,980] Using an existing study with name 'catboost_heart_disease_opt' instead of creating a new one.


📊 Trạng thái hiện tại: Đã chạy xong 21 trials.
🚀 Bắt đầu chạy tiếp 29 trials còn lại...


...

[I 2026-03-24 15:12:16,011] Trial 27 finished with value: 0.9554732108488657 and parameters: {'learning_rate': 0.07111335358593021, 'depth': 5, 'l2_leaf_reg': 3.869149510713656, 'random_strength': 2.558895145611108, 'border_count': 254}. Best is trial 27 with value: 0.9554732108488657.


☁️ Đang đồng bộ trạng thái sau Trial 27 lên HF...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Đã lưu tiến độ an toàn!


**STRATIFIED K-FOLD CROSS VALIDATION**

In [11]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

print("Bắt đầu huấn luyện...")
for fold, (train_idx, valid_idx) in enumerate(skf.split(train[features], train[TARGET])):
    X_train, y_train = train.iloc[train_idx][features], train.iloc[train_idx][TARGET]
    X_valid, y_valid = train.iloc[valid_idx][features], train.iloc[valid_idx][TARGET]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.09810457243812704,
        depth=6,
        l2_leaf_reg=4.034076521920937,
        random_strength=3.129412805235088,
        border_count=128,
        eval_metric='AUC',
        random_seed=42,
        cat_features=cat_features,
        verbose=100
    )
    
    model.fit(
        X_train, y_train,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=100,
        use_best_model=True
    )
    
    oof_preds[valid_idx] = model.predict_proba(X_valid)[:, 1]
    
    test_preds += model.predict_proba(test[features])[:, 1] / N_SPLITS
    
    fold_auc = roc_auc_score(y_valid, oof_preds[valid_idx])
    print(f"Fold {fold+1} AUC: {fold_auc:.4f}\n")


Bắt đầu huấn luyện...
0:	test: 0.9243733	best: 0.9243733 (0)	total: 220ms	remaining: 3m 39s
100:	test: 0.9544777	best: 0.9544777 (100)	total: 19.2s	remaining: 2m 51s
200:	test: 0.9552110	best: 0.9552110 (200)	total: 38.8s	remaining: 2m 34s
300:	test: 0.9555267	best: 0.9555267 (300)	total: 59.5s	remaining: 2m 18s
400:	test: 0.9556602	best: 0.9556603 (399)	total: 1m 20s	remaining: 1m 59s
500:	test: 0.9557518	best: 0.9557520 (499)	total: 1m 41s	remaining: 1m 41s
600:	test: 0.9557779	best: 0.9557786 (599)	total: 2m 2s	remaining: 1m 21s
700:	test: 0.9557852	best: 0.9557886 (693)	total: 2m 23s	remaining: 1m 1s
800:	test: 0.9557855	best: 0.9557938 (757)	total: 2m 45s	remaining: 41.1s
900:	test: 0.9557819	best: 0.9557938 (836)	total: 3m 7s	remaining: 20.6s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9557938245
bestIteration = 836

Shrink model to first 837 iterations.
Fold 1 AUC: 0.9558

0:	test: 0.9232082	best: 0.9232082 (0)	total: 223ms	remaining: 3m 43s
100:	test: 0

**OUTPUT SUBMISSION**

In [12]:
cv_auc = roc_auc_score(train[TARGET], oof_preds)
print(f"Tổng thể Cross Validation AUC: {cv_auc:.4f}")

submission = pd.DataFrame({
    'id': test['id'],
    'Heart Disease': test_preds
})

submission.to_csv('submission.csv', index=False)
print("Đã tạo file submission.csv thành công!")

Tổng thể Cross Validation AUC: 0.9554
Đã tạo file submission.csv thành công!
